# Realist ERCOT — DC-SCED Test Run

**Target:** 2025-11-05 (November weekday, high West Texas wind)

**Grid:** 4,303 OSM buses · 4,679 transmission branches · 1,174 generators (465 thermal, 709 renewable)

## Upload to cluster (same directory as this notebook)

```
SourceData/
    bus.csv           ← 4,303 OSM buses
    branch.csv        ← 4,679 transmission branches
    gen.csv           ← 1,174 generators
    init_state.csv    ← initial commitment state
run_realist_sced.ipynb
```

## Cluster environment (Adroit)

```bash
conda activate vatic-test   # same env as precept_testfile
# Solver: gurobi (licensed on Adroit)
```


In [ ]:
import sys, math, datetime, warnings
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# Vatic is installed in the cluster env (conda activate vatic-test)
from vatic.data.loaders import GridLoader
from vatic.engines import Simulator

HERE    = Path(".").resolve()
SRC_DIR = HERE / "SourceData"
for fname in ("bus.csv", "branch.csv", "gen.csv", "init_state.csv"):
    assert (SRC_DIR / fname).exists(), f"Missing SourceData/{fname}"
print("All input files found.")
print(f"  bus.csv:        {sum(1 for _ in open(SRC_DIR/'bus.csv'))-1:,} rows")
print(f"  branch.csv:     {sum(1 for _ in open(SRC_DIR/'branch.csv'))-1:,} rows")
print(f"  gen.csv:        {sum(1 for _ in open(SRC_DIR/'gen.csv'))-1:,} rows")
print(f"  init_state.csv: {sum(1 for _ in open(SRC_DIR/'init_state.csv'))-1:,} rows")


## Step 1 — Distribute zonal load to buses (Census 2020 county population weights)

ERCOT zone totals (November 2025-11-05 17:00 CST estimate):

| Zone    | MW     |
|---------|--------|
| NORTH   | 23,000 |
| HOUSTON | 11,500 |
| SOUTH   | 10,500 |
| WEST    |  7,000 |

Each bus's share is proportional to the Census 2020 population of its nearest Texas county centroid.

In [ ]:
# ---------------------------------------------------------------------------
# Zonal load targets (MW) and DC tie fixed imports
# ---------------------------------------------------------------------------
ZONE_LOAD_MW = {
    'NORTH':   23_000.0,
    'HOUSTON': 11_500.0,
    'SOUTH':   10_500.0,
    'WEST':     7_000.0,
}

DC_TIES = [
    ('OKLAUNION',  220.0),
    ('MONTICELLO', 600.0),
    ('EAGLE',       36.0),
    ('MCALLEN',    150.0),
    ('LAREDO',     100.0),
]

# ---------------------------------------------------------------------------
# Texas county populations (Census 2020) and approximate centroids
# 254 counties; (centroid_lat, centroid_lon, population_2020)
# ---------------------------------------------------------------------------
TEXAS_COUNTY_POP = {
    'Anderson':      (31.82,  -95.65,   57_863),
    'Andrews':       (32.31, -102.64,   18_705),
    'Angelina':      (31.37,  -94.62,   86_771),
    'Aransas':       (28.12,  -97.05,   23_510),
    'Archer':        (33.62,  -98.69,    8_474),
    'Armstrong':     (34.97, -101.36,    1_848),
    'Atascosa':      (28.89,  -98.53,   48_781),
    'Austin':        (29.89,  -96.28,   30_167),
    'Bailey':        (34.07, -102.83,    6_985),
    'Bandera':       (29.75,  -99.25,   21_941),
    'Bastrop':       (30.10,  -97.31,   97_216),
    'Baylor':        (33.62,  -99.22,    3_530),
    'Bee':           (28.42,  -97.74,   32_691),
    'Bell':          (31.05,  -97.48,  362_924),
    'Bexar':         (29.45,  -98.52, 2_009_324),
    'Blanco':        (30.26,  -98.41,   11_279),
    'Borden':        (32.74, -101.43,      641),
    'Bosque':        (31.90,  -97.64,   18_685),
    'Bowie':         (33.44,  -94.16,   94_090),
    'Brazoria':      (29.17,  -95.49,  372_031),
    'Brazos':        (30.66,  -96.30,  229_211),
    'Brewster':      (29.79, -103.25,    9_203),
    'Briscoe':       (34.53, -101.20,    1_546),
    'Brooks':        (27.03,  -98.22,    7_076),
    'Brown':         (31.77,  -99.00,   37_864),
    'Burleson':      (30.49,  -96.61,   18_443),
    'Burnet':        (30.79,  -98.23,   47_597),
    'Caldwell':      (29.83,  -97.62,   45_883),
    'Calhoun':       (28.44,  -96.61,   21_290),
    'Callahan':      (32.30,  -99.37,   13_943),
    'Cameron':       (26.15,  -97.58,  423_163),
    'Camp':          (33.00,  -94.98,   13_094),
    'Carson':        (35.40, -101.35,    5_926),
    'Cass':          (33.07,  -94.34,   30_016),
    'Castro':        (34.53, -102.26,    7_530),
    'Chambers':      (29.71,  -94.63,   45_689),
    'Cherokee':      (31.83,  -95.17,   52_646),
    'Childress':     (34.53, -100.21,    7_306),
    'Clay':          (33.78,  -98.20,   10_303),
    'Cochran':       (33.60, -102.84,    2_547),
    'Coke':          (31.89, -100.52,    3_009),
    'Coleman':       (31.77,  -99.43,    8_547),
    'Collin':        (33.19,  -96.57, 1_064_465),
    'Collingsworth': (34.96, -100.27,    2_920),
    'Colorado':      (29.62,  -96.53,   21_493),
    'Comal':         (29.82,  -98.27,  156_209),
    'Comanche':      (31.95,  -98.56,   13_635),
    'Concho':        (31.32,  -99.74,    2_726),
    'Cooke':         (33.64,  -97.21,   41_071),
    'Coryell':       (31.39,  -97.79,   80_766),
    'Cottle':        (34.08, -100.28,    1_398),
    'Crane':         (31.43, -102.35,    4_797),
    'Crockett':      (30.72, -101.42,    3_405),
    'Crosby':        (33.61, -101.30,    5_737),
    'Culberson':     (31.44, -104.52,    2_163),
    'Dallam':        (36.28, -102.60,    6_703),
    'Dallas':        (32.77,  -96.80, 2_613_539),
    'Dawson':        (32.74, -101.95,   12_547),
    'Deaf Smith':    (34.96, -102.60,   18_546),
    'Delta':         (33.39,  -95.68,    5_331),
    'Denton':        (33.21,  -97.13,  906_422),
    'DeWitt':        (29.09,  -97.35,   20_097),
    'Dickens':       (33.62, -100.79,    2_211),
    'Dimmit':        (28.43,  -99.75,   10_124),
    'Donley':        (34.96, -100.81,    3_278),
    'Duval':         (27.68,  -98.49,   11_157),
    'Eastland':      (32.31,  -98.82,   18_583),
    'Ector':         (31.87, -102.53,  166_223),
    'Edwards':       (29.98, -100.30,    1_932),
    'El Paso':       (31.77, -106.49,  865_657),
    'Ellis':         (32.35,  -96.76,  185_141),
    'Erath':         (32.23,  -98.20,   43_564),
    'Falls':         (31.27,  -96.93,   17_297),
    'Fannin':        (33.59,  -96.11,   36_496),
    'Fayette':       (29.88,  -96.92,   25_066),
    'Fisher':        (32.74, -100.40,    3_848),
    'Floyd':         (33.97, -101.30,    5_728),
    'Foard':         (33.98,  -99.78,    1_186),
    'Fort Bend':     (29.53,  -95.77,  811_688),
    'Franklin':      (33.17,  -95.22,   10_720),
    'Freestone':     (31.70,  -96.15,   19_717),
    'Frio':          (28.87,  -99.11,   20_306),
    'Gaines':        (32.74, -102.63,   22_010),
    'Galveston':     (29.37,  -94.85,  342_139),
    'Garza':         (33.18, -101.30,    6_229),
    'Gillespie':     (30.32,  -98.94,   26_208),
    'Glasscock':     (31.87, -101.52,    1_408),
    'Goliad':        (28.66,  -97.45,    7_658),
    'Gonzales':      (29.46,  -97.49,   20_837),
    'Gray':          (35.40, -100.81,   21_886),
    'Grayson':       (33.62,  -96.68,  136_212),
    'Gregg':         (32.47,  -94.82,  123_945),
    'Grimes':        (30.54,  -95.93,   28_880),
    'Guadalupe':     (29.61,  -97.96,  166_847),
    'Hale':          (34.07, -101.82,   33_406),
    'Hall':          (34.53, -100.68,    2_964),
    'Hamilton':      (31.69,  -98.11,    8_461),
    'Hansford':      (36.28, -101.35,    5_399),
    'Hardeman':      (34.29,  -99.75,    3_801),
    'Hardin':        (30.27,  -94.36,   57_602),
    'Harris':        (29.85,  -95.40, 4_731_145),
    'Harrison':      (32.55,  -94.38,   66_553),
    'Hartley':       (35.84, -102.60,    5_576),
    'Haskell':       (33.18,  -99.73,    5_336),
    'Hays':          (30.06,  -98.03,  246_521),
    'Hemphill':      (35.84, -100.27,    3_819),
    'Henderson':     (32.22,  -95.85,   82_737),
    'Hidalgo':       (26.40,  -98.10,  870_781),
    'Hill':          (31.99,  -97.13,   35_399),
    'Hockley':       (33.61, -102.35,   23_006),
    'Hood':          (32.44,  -97.82,   64_099),
    'Hopkins':       (33.15,  -95.56,   37_084),
    'Houston':       (31.32,  -95.42,   22_968),
    'Howard':        (32.31, -101.44,   36_664),
    'Hudspeth':      (31.46, -105.38,    4_886),
    'Hunt':          (33.13,  -96.09,   99_630),
    'Hutchinson':    (35.84, -101.35,   21_061),
    'Irion':         (31.32, -100.98,    1_536),
    'Jack':          (33.23,  -98.17,    9_003),
    'Jackson':       (28.96,  -96.58,   14_591),
    'Jasper':        (30.72,  -93.99,   35_710),
    'Jeff Davis':    (30.72, -104.12,    2_274),
    'Jefferson':     (30.04,  -94.17,  252_358),
    'Jim Hogg':      (27.06,  -99.08,    5_300),
    'Jim Wells':     (27.73,  -98.08,   40_128),
    'Johnson':       (32.38,  -97.37,  179_685),
    'Jones':         (32.74,  -99.87,   19_891),
    'Karnes':        (28.89,  -97.86,   15_505),
    'Kaufman':       (32.60,  -96.28,  136_154),
    'Kendall':       (29.95,  -98.70,   46_687),
    'Kenedy':        (26.93,  -97.65,      404),
    'Kent':          (33.18, -100.77,      762),
    'Kerr':          (30.06,  -99.34,   53_635),
    'Kimble':        (30.50,  -99.74,    4_472),
    'King':          (33.62, -100.26,      272),
    'Kinney':        (29.35, -100.42,    3_667),
    'Kleberg':       (27.43,  -97.81,   31_549),
    'Knox':          (33.60,  -99.76,    3_664),
    'La Salle':      (28.34,  -99.10,    7_430),
    'Lamar':         (33.67,  -95.54,   49_532),
    'Lamb':          (34.07, -102.35,   13_262),
    'Lampasas':      (31.19,  -98.24,   21_281),
    'Lavaca':        (29.38,  -96.92,   20_154),
    'Lee':           (30.32,  -97.04,   17_239),
    'Leon':          (31.29,  -95.97,   17_151),
    'Liberty':       (30.17,  -94.82,   90_697),
    'Limestone':     (31.54,  -96.59,   23_437),
    'Lipscomb':      (36.28, -100.27,    3_233),
    'Live Oak':      (28.35,  -98.12,   12_207),
    'Llano':         (30.71,  -98.69,   20_860),
    'Loving':        (31.85, -103.59,       64),
    'Lubbock':       (33.61, -101.82,  310_569),
    'Lynn':          (33.18, -101.82,    5_808),
    'Madison':       (30.97,  -95.92,   14_218),
    'Marion':        (33.00,  -94.36,   10_083),
    'Martin':        (32.31, -101.95,    5_771),
    'Mason':         (30.73,  -99.23,    4_274),
    'Matagorda':     (28.79,  -96.01,   36_702),
    'Maverick':      (28.74, -100.31,   57_887),
    'McCulloch':     (31.20,  -99.34,    7_984),
    'McLennan':      (31.55,  -97.17,  262_065),
    'McMullen':      (28.35,  -98.57,      707),
    'Medina':        (29.35,  -99.11,   50_607),
    'Menard':        (30.88,  -99.82,    2_148),
    'Midland':       (32.00, -102.08,  169_895),
    'Milam':         (30.79,  -96.97,   24_823),
    'Mills':         (31.49,  -98.60,    4_873),
    'Mitchell':      (32.31, -100.92,    8_545),
    'Montague':      (33.67,  -97.73,   19_546),
    'Montgomery':    (30.30,  -95.50,  620_443),
    'Moore':         (35.84, -101.89,   21_904),
    'Morris':        (33.11,  -94.71,   12_388),
    'Motley':        (34.07, -100.79,    1_156),
    'Nacogdoches':   (31.62,  -94.65,   64_785),
    'Navarro':       (32.05,  -96.47,   50_125),
    'Newton':        (30.77,  -93.73,   13_488),
    'Nolan':         (32.31, -100.40,   14_669),
    'Nueces':        (27.73,  -97.59,  342_510),
    'Ochiltree':     (36.28, -100.81,    9_836),
    'Oldham':        (35.40, -102.60,    1_911),
    'Orange':        (30.13,  -93.86,   84_047),
    'Palo Pinto':    (32.74,  -98.30,   28_409),
    'Panola':        (32.15,  -94.31,   23_440),
    'Parker':        (32.77,  -97.81,  148_222),
    'Parmer':        (34.53, -102.78,    9_605),
    'Pecos':         (30.79, -102.72,   15_823),
    'Polk':          (30.82,  -94.83,   51_353),
    'Potter':        (35.40, -101.88,  117_415),
    'Presidio':      (29.79, -104.35,    6_131),
    'Rains':         (32.87,  -95.79,   12_514),
    'Randall':       (34.96, -101.89,  140_977),
    'Reagan':        (31.37, -101.52,    3_367),
    'Real':          (29.83,  -99.83,    3_389),
    'Red River':     (33.63,  -94.99,   12_023),
    'Reeves':        (31.32, -103.69,   15_976),
    'Refugio':       (28.33,  -97.16,    7_236),
    'Roberts':       (35.84, -100.81,      885),
    'Robertson':     (31.02,  -96.51,   16_953),
    'Rockwall':      (32.92,  -96.41,  107_741),
    'Runnels':       (31.83,  -99.97,   10_264),
    'Rusk':          (32.11,  -94.77,   53_595),
    'Sabine':        (31.35,  -93.87,   10_542),
    'San Augustine': (31.39,  -94.17,    8_490),
    'San Jacinto':   (30.57,  -95.10,   29_773),
    'San Patricio':  (27.97,  -97.52,   67_138),
    'San Saba':      (31.17,  -98.72,    6_055),
    'Schleicher':    (30.90, -100.54,    2_793),
    'Scurry':        (32.74, -100.91,   16_703),
    'Shackelford':   (32.74,  -99.35,    3_282),
    'Shelby':        (31.79,  -94.14,   25_048),
    'Sherman':       (36.28, -101.89,    3_034),
    'Smith':         (32.38,  -95.27,  232_751),
    'Somervell':     (32.22,  -97.77,    9_128),
    'Starr':         (26.56,  -98.77,   64_633),
    'Stephens':      (32.73,  -98.82,    9_366),
    'Sterling':      (31.83, -101.05,    1_291),
    'Stonewall':     (33.18, -100.25,    1_285),
    'Sutton':        (30.51, -100.53,    3_786),
    'Swisher':       (34.53, -101.74,    7_236),
    'Tarrant':       (32.77,  -97.29, 2_110_640),
    'Taylor':        (32.31,  -99.89,  138_034),
    'Terrell':       (30.22, -102.08,      775),
    'Terry':         (33.18, -102.35,   12_004),
    'Throckmorton':  (33.18,  -99.21,    1_517),
    'Titus':         (33.21,  -94.96,   32_750),
    'Tom Green':     (31.40, -100.45,  119_664),
    'Travis':        (30.33,  -97.77, 1_290_188),
    'Trinity':       (31.09,  -95.37,   14_585),
    'Tyler':         (30.77,  -94.35,   21_672),
    'Upshur':        (32.73,  -94.96,   41_782),
    'Upton':         (31.37, -102.05,    3_657),
    'Uvalde':        (29.36,  -99.78,   25_926),
    'Val Verde':     (29.89, -101.15,   48_879),
    'Van Zandt':     (32.56,  -95.83,   56_590),
    'Victoria':      (28.80,  -96.98,   92_084),
    'Walker':        (30.74,  -95.57,   72_791),
    'Waller':        (30.00,  -95.99,   55_246),
    'Ward':          (31.51, -103.10,   11_998),
    'Washington':    (30.21,  -96.39,   34_796),
    'Webb':          (27.74,  -99.51,  276_652),
    'Wharton':       (29.31,  -96.21,   41_551),
    'Wheeler':       (35.40, -100.27,    5_056),
    'Wichita':       (33.99,  -98.71,  131_818),
    'Wilbarger':     (34.09,  -99.25,   12_769),
    'Willacy':       (26.47,  -97.82,   20_880),
    'Williamson':    (30.65,  -97.60,  609_017),
    'Wilson':        (29.18,  -98.07,   51_584),
    'Winkler':       (31.85, -103.06,    7_802),
    'Wise':          (33.21,  -97.65,   77_028),
    'Wood':          (32.78,  -95.38,   45_539),
    'Yoakum':        (33.18, -102.82,    8_713),
    'Young':         (33.17,  -98.68,   17_806),
    'Zapata':        (27.07,  -99.17,   14_179),
    'Zavala':        (28.86,  -99.76,   12_166),
}
print(f'Counties loaded: {len(TEXAS_COUNTY_POP)}')

In [ ]:
# ---------------------------------------------------------------------------
# Helper functions: haversine distance, nearest county, load distribution
# ---------------------------------------------------------------------------
_COUNTY_LIST = list(TEXAS_COUNTY_POP.values())  # (lat, lon, pop) tuples

def _haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2)**2
         + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2))
         * math.sin(dlon / 2)**2)
    return R * 2 * math.asin(math.sqrt(a))

def _nearest_county_pop(lat, lon):
    best_dist, best_pop = float('inf'), 1
    for clat, clon, pop in _COUNTY_LIST:
        d = _haversine_km(lat, lon, clat, clon)
        if d < best_dist:
            best_dist, best_pop = d, pop
    return best_pop

def distribute_load(bus_csv_path):
    """
    Distribute zone MW load proportional to Census 2020 county population.
    Applies DC tie adjustments. Overwrites the bus.csv file in place.
    """
    bus = pd.read_csv(bus_csv_path, dtype=str)
    bus['_lat'] = pd.to_numeric(bus['lat'], errors='coerce')
    bus['_lon'] = pd.to_numeric(bus['lng'], errors='coerce')
    bus['MW Load'] = 0.0

    pop_weights = []
    for _, row in bus.iterrows():
        lat, lon = row['_lat'], row['_lon']
        if pd.isna(lat) or pd.isna(lon):
            pop_weights.append(1.0)
        else:
            pop_weights.append(float(_nearest_county_pop(float(lat), float(lon))))
    bus['_pop'] = pop_weights

    zone_total_pop = bus.groupby('Zone')['_pop'].sum().to_dict()

    load_col = []
    for _, row in bus.iterrows():
        zone    = str(row.get('Zone', '')).strip()
        zone_mw = ZONE_LOAD_MW.get(zone, 0.0)
        ztotal  = zone_total_pop.get(zone, 0.0)
        pop     = row['_pop']
        load_col.append(round(zone_mw * pop / ztotal, 6) if ztotal > 0 else 0.0)
    bus['MW Load'] = load_col

    sub_name_col = bus['Sub Name'].fillna('').str.upper()
    for keyword, import_mw in DC_TIES:
        matches = bus.index[sub_name_col.str.contains(keyword, regex=False)]
        if len(matches) == 0:
            matches = bus.index[bus['Zone'] == 'SOUTH']
        if len(matches) > 0:
            idx = matches[0]
            bus.loc[idx, 'MW Load'] = float(bus.loc[idx, 'MW Load']) - import_mw

    bus = bus.drop(columns=['_lat', '_lon', '_pop'])
    bus.to_csv(bus_csv_path, index=False)
    return bus

In [ ]:
print('Distributing load (may take ~30 seconds for 4,303 buses × 254 counties)...')
bus_df = distribute_load(SRC_DIR / 'bus.csv')

print('Zone load totals after distribution:')
bus_df['_load'] = pd.to_numeric(bus_df['MW Load'], errors='coerce').fillna(0.0)
zone_totals = bus_df.groupby('Zone')['_load'].sum()
for zone, mw in zone_totals.items():
    print(f'  {zone:10s}: {mw:>10,.1f} MW')
print(f'  {"TOTAL":10s}: {zone_totals.sum():>10,.1f} MW')

## Step 2 — Define RealistLoader

Subclasses `GridLoader` from the Vatic library. Reads the pre-built CSVs in `SourceData/`
and provides a constant 48-hour timeseries (two days, as Vatic requires).

In [ ]:
class RealistLoader(GridLoader):
    """Custom GridLoader for the Realist ERCOT model."""

    grid_lbl = 'Realist'
    _data_dir = 'SourceData'

    thermal_gen_types = {'Nuclear': 'N', 'Coal': 'C', 'Gas': 'G'}
    renew_gen_types   = {'Wind': 'W', 'Solar': 'S'}

    @property
    def data_path(self):
        # GridLoader reads from data_path/SourceData/*.csv
        return HERE

    @property
    def init_state_file(self):
        return SRC_DIR / 'init_state.csv'

    @property
    def utc_offset(self):
        return -pd.Timedelta(hours=6)   # CST = UTC-6

    @property
    def timeseries_cohorts(self):
        return {'Wind', 'Solar'}

    @staticmethod
    def get_dispatch_types(renew_types):
        return {
            'DispatchRenewables':    set(renew_types),
            'NondispatchRenewables': set(),
            'ForecastRenewables':    set(renew_types),
        }

    @staticmethod
    def process_actuals(actuals_file, start_date=None, end_date=None):
        return pd.DataFrame()

    @classmethod
    def must_gen_run(cls, gen):
        return gen.Fuel == 'Nuclear'

    def get_generator_type(self, gen):
        row = self.gen_df.loc[self.gen_df['GEN UID'] == gen]
        if row.empty:
            return 'G'
        fuel = str(row['Fuel'].iloc[0])
        if fuel == 'Wind':  return 'WIND'
        if fuel == 'Solar': return 'PV'
        return 'G'

    def get_generator_zone(self, gen):
        row = self.gen_df.loc[self.gen_df['GEN UID'] == gen]
        if row.empty:
            return 'NORTH'
        bus_id = row['Bus ID'].iloc[0]
        bus_row = self.bus_df.loc[self.bus_df['Bus ID'].astype(str) == str(bus_id)]
        return 'NORTH' if bus_row.empty else str(bus_row['Zone'].iloc[0])

    @classmethod
    def parse_generator(cls, gen_info):
        import math as _math
        break_cols = [c for c in gen_info.index if c.startswith('MW Break')]
        price_cols = [c for c in gen_info.index if c.startswith('MWh Price')]
        break_idxs = {int(c.split(' Break ')[1]): c for c in break_cols}
        price_idxs = {int(c.split(' Price ')[1]): c for c in price_cols}
        cost_idxs  = sorted(set(break_idxs) & set(price_idxs))

        if not cost_idxs:
            cost_values = [0.0, 0.0]
        else:
            cost_values = [
                float(gen_info['Fixed Cost($/hr)'])
                + float(gen_info[price_idxs[cost_idxs[0]]])
                * float(gen_info[break_idxs[cost_idxs[0]]])
            ]
            for i in range(len(cost_idxs) - 1):
                cost_values.append(
                    cost_values[-1]
                    + float(gen_info[price_idxs[cost_idxs[i]]])
                    * (float(gen_info[break_idxs[cost_idxs[i+1]]])
                       - float(gen_info[break_idxs[cost_idxs[i]]]))
                )
            cost_values.append(
                cost_values[-1]
                + float(gen_info[price_idxs[cost_idxs[-1]]])
                * (float(gen_info['PMax MW'])
                   - float(gen_info[break_idxs[cost_idxs[-1]]]))
            )

        cost_points = [float(gen_info[c]) for c in break_cols] + [float(gen_info['PMax MW'])]

        return cls.Generator(
            gen_info['GEN UID'],
            int(float(gen_info['Bus ID'])),
            gen_info['Unit Group'],
            gen_info['Unit Type'],
            gen_info['Fuel'],
            float(gen_info['PMin MW']),
            float(gen_info['PMax MW']),
            int(_math.ceil(float(gen_info['Min Down Time Hr']))),
            int(_math.ceil(float(gen_info['Min Up Time Hr']))),
            float(gen_info['Ramp Rate MW/Min']),
            int(float(gen_info['Start Time Cold Hr'])),
            int(float(gen_info['Start Time Warm Hr'])),
            int(float(gen_info['Start Time Hot Hr'])),
            float(gen_info['Start Heat Cold MBTU']),
            float(gen_info['Start Heat Warm MBTU']),
            float(gen_info['Start Heat Hot MBTU']),
            float(gen_info['Fuel Price $/MMBTU']),
            cost_points,
            cost_values,
        )

    def create_timeseries(self, start_date=None, end_date=None):
        """
        48-hour constant timeseries (2025-11-05 + 2025-11-06 UTC).
        Renewable generators: constant PMax (capacity factor already applied).
        Buses: constant MW Load (set by distribute_load).
        """
        times = pd.date_range('2025-11-05T00:00:00', periods=48, freq='h', tz='UTC')

        renew_gens = [g for g in self.generators if g.Fuel in self.renew_gen_types]
        gen_cols = {}
        for g in renew_gens:
            for tag in ('fcst', 'actl'):
                gen_cols[(tag, g.ID)] = [round(g.MaxPower, 4)] * 48
        if gen_cols:
            gen_df = pd.DataFrame(gen_cols, index=times)
            gen_df.columns = pd.MultiIndex.from_tuples(gen_df.columns)
        else:
            gen_df = pd.DataFrame(index=times)
            gen_df.columns = pd.MultiIndex.from_tuples([], names=[None, None])

        load_cols = {}
        bus_name_col = self.bus_df['Bus Name'].fillna('')
        mw_load_col  = pd.to_numeric(self.bus_df['MW Load'], errors='coerce').fillna(0.0)
        for bus_name, mw_load in zip(bus_name_col, mw_load_col):
            bname = str(bus_name).strip()
            if not bname:
                continue
            for tag in ('fcst', 'actl'):
                load_cols[(tag, bname)] = [round(float(mw_load), 4)] * 48
        load_df = pd.DataFrame(load_cols, index=times)
        load_df.columns = pd.MultiIndex.from_tuples(load_df.columns)
        load_df = load_df.sort_index(axis=1)

        return gen_df, load_df

## Step 3 — Load grid and build timeseries

In [ ]:
print('Initializing RealistLoader...')
loader = RealistLoader()

n_thermal  = sum(1 for g in loader.generators if g.Fuel in loader.thermal_gen_types)
n_renew    = sum(1 for g in loader.generators if g.Fuel in loader.renew_gen_types)
n_branches = len(loader.branches)
n_buses    = len(loader.buses)

print(f'  Buses:              {n_buses:,}')
print(f'  Branches:           {n_branches:,}')
print(f'  Thermal generators: {n_thermal:,}')
print(f'  Renewable gens:     {n_renew:,}')

assert n_branches > 0, 'No branches loaded — check branch.csv'
assert n_thermal  > 0, 'No thermal generators — check gen.csv'

gen_df, load_df = loader.create_timeseries()
print(f'  gen_data shape:     {gen_df.shape}   (renewable gens × 48 h, 2 tags)')
print(f'  load_data shape:    {load_df.shape}  (buses × 48 h, 2 tags)')

## Step 4 — Run Vatic Simulator

Uses **gurobi** with 8 threads (Adroit licence). Set `RUN_LMPS=False` for a faster first run.

Expected runtime: **5–15 minutes** on Adroit.


In [ ]:
SOLVER      = 'gurobi'    # gurobi is licensed on Adroit
RUN_LMPS    = True         # False for faster run; True to see price spreads
MIPGAP      = 0.01         # 1% optimality gap
N_THREADS   = 8            # Adroit allocation

solver_opts = {'Threads': N_THREADS} if SOLVER == 'gurobi' else {}

print(f'Starting Simulator | solver={SOLVER} | LMPs={RUN_LMPS} | mipgap={MIPGAP}')

sim = Simulator(
    template_data=loader.template,
    gen_data=gen_df,
    load_data=load_df,
    out_dir=None,
    start_date=datetime.date(2025, 11, 5),
    num_days=1,
    solver=SOLVER,
    solver_options=solver_opts,
    run_lmps=RUN_LMPS,
    mipgap=MIPGAP,
    load_shed_penalty=1e4,
    reserve_shortfall_penalty=1e3,
    reserve_factor=0.05,
    output_detail=3,
    prescient_sced_forecasts=True,
    ruc_prescience_hour=0,
    ruc_execution_hour=16,
    ruc_every_hours=24,
    ruc_horizon=24,
    sced_horizon=1,
    lmp_shortfall_costs=False,
    enforce_sced_shutdown_ramprate=False,
    no_startup_shutdown_curves=False,
    init_ruc_file=None,
    verbosity=1,
    output_max_decimals=4,
    create_plots=False,
    renew_costs=None,
    save_to_csv=False,
    last_conditions_file=None,
)

report_dfs = sim.simulate()
print('\nSimulation complete.')

## Step 5 — Results

In [ ]:
print('Available report keys:')
print(list(report_dfs.keys()))

In [ ]:
# Hourly system summary
report_dfs['hourly_summary'].head()

In [ ]:
# Generation by fuel type
if 'thermal_detail' in report_dfs:
    th = report_dfs['thermal_detail']
    fuel_col = next((c for c in th.columns if 'fuel' in c.lower()), None)
    mw_col   = next((c for c in th.columns if 'dispatch' in c.lower() or 'output' in c.lower()), None)
    if fuel_col and mw_col:
        th[mw_col] = pd.to_numeric(th[mw_col], errors='coerce')
        by_fuel = th.groupby(fuel_col)[mw_col].mean().sort_values(ascending=False)
        print('Average dispatch by fuel (MW):')
        for fuel, mw in by_fuel.items():
            print(f'  {fuel:12s}: {mw:>10,.1f} MW')
    else:
        display(th.head())

In [ ]:
# Bus LMP analysis (requires RUN_LMPS=True)
if RUN_LMPS and 'bus_detail' in report_dfs:
    bd = report_dfs['bus_detail'].copy()
    lmp_col = next((c for c in bd.columns if 'lmp' in c.lower()), None)
    bus_col = next((c for c in bd.columns if 'bus' in c.lower()), None)

    if lmp_col and bus_col:
        bd[lmp_col] = pd.to_numeric(bd[lmp_col], errors='coerce')
        bus_avg = bd.groupby(bus_col)[lmp_col].mean().dropna()

        print('Top 10 most expensive buses (avg LMP $/MWh):')
        for b, lmp in bus_avg.nlargest(10).items():
            print(f'  {str(b):40s}: ${lmp:8.2f}')

        print('\nTop 10 cheapest buses (avg LMP $/MWh):')
        for b, lmp in bus_avg.nsmallest(10).items():
            print(f'  {str(b):40s}: ${lmp:8.2f}')

        # Zone averages
        bus_zone = (
            pd.read_csv(SRC_DIR / 'bus.csv', dtype=str)[['Bus Name', 'Zone']]
            .set_index('Bus Name')['Zone'].to_dict()
        )
        bus_avg_df = bus_avg.reset_index()
        bus_avg_df.columns = ['Bus Name', 'LMP']
        bus_avg_df['Zone'] = bus_avg_df['Bus Name'].map(bus_zone)
        zone_avg = bus_avg_df.groupby('Zone')['LMP'].mean()

        print('\nZone average LMPs:')
        for z, lmp in zone_avg.items():
            print(f'  {z:10s}: ${lmp:8.2f}/MWh')

        west  = zone_avg.get('WEST',  float('nan'))
        north = zone_avg.get('NORTH', float('nan'))
        if not (math.isnan(west) or math.isnan(north)):
            print(f'\nWest–North spread: ${north - west:.2f}/MWh')
    else:
        display(bd.head())
else:
    print('LMPs not computed (set RUN_LMPS=True to enable).')

In [ ]:
# Transmission line loading
if 'line_detail' in report_dfs:
    ld = report_dfs['line_detail'].copy()
    flow_col = next((c for c in ld.columns if 'flow' in c.lower()), None)
    if flow_col:
        ld[flow_col] = pd.to_numeric(ld[flow_col], errors='coerce').abs()
        print('Top 10 most loaded lines (avg |MW flow|):')
        line_col = ld.columns[0]
        top_lines = ld.groupby(line_col)[flow_col].mean().nlargest(10)
        for line, mw in top_lines.items():
            print(f'  {str(line):45s}: {mw:>8,.1f} MW')
    else:
        display(ld.head())